# **Classificação dos pacientes conforme o percentual da meta energética (meta_kcal) atingido durante a internação**.


## Cria duas novas colunas no DataFrame:

    - TresGruposKcal: ProxMeta / MetMeta / AbaixoMeta

    - DoisGruposKcal: ProxMeta / MetMeta


In [325]:
import numpy as np
import pandas as pd

# Passo 1 - Ler o banco de dados

In [326]:
df = pd.read_excel("data/data.xlsx")
df.shape

(57, 49)

# Passo 2 - Identificar as colunas de consumo diário (cons_kcal_day1, day2, ...)

In [327]:
# A expressão regular garante que apenas colunas no formato "cons_kcal_dayX"
# (X = número do dia) sejam selecionadas.
colunas_dias = df.filter(regex=r"^cons_kcal_day\d+$").columns

# Ordena as colunas pelo número do dia (day1, day2, ..., day10, ...),
# evitando a ordem alfabética (day1, day10, day11, ..., day2).
colunas_dias = sorted(colunas_dias, key=lambda c: int(c.replace("cons_kcal_day", "")))


# Passo 3 - Garantir que os valores sejam numéricos

In [328]:
# Qualquer célula que não contenha número (vazia, texto etc.) vira NaN.
# Assim, os "dias internados" são exatamente as células com números.
consumo = df[colunas_dias].apply(pd.to_numeric, errors="coerce")
meta = pd.to_numeric(df["meta_kcal"], errors="coerce")


# Passo 4 - Calcular o percentual da meta atingido em cada dia

In [329]:
# Divide o consumo de cada dia pela meta do respectivo paciente (linha a linha).
# Dias sem dado continuam como NaN.
pct_meta = consumo.div(meta, axis=0) * 100

# Passo 5 - Contar os dias internados de cada paciente

In [330]:
# Número de colunas cons_kcal_dayX que contêm números.
dias_internados = pct_meta.notna().sum(axis=1)


# Passo 6 - Calcular a proporção de dias em cada faixa de consumo

In [331]:
# Comparações com NaN retornam False, portanto dias sem dado não são contados.
dias_70_ou_mais = (pct_meta >= 70).sum(axis=1)                        # >= 70%
dias_50_a_69 = ((pct_meta >= 50) & (pct_meta < 70)).sum(axis=1)       # 50% a 69%
dias_50_ou_mais = (pct_meta >= 50).sum(axis=1)                        # >= 50%

# Proporção de dias (0 a 1). Pacientes sem nenhum dia registrado ficam NaN.
prop_70_ou_mais = dias_70_ou_mais / dias_internados.replace(0, np.nan)
prop_50_ou_mais = dias_50_ou_mais / dias_internados.replace(0, np.nan)


# Passo 7 - Criar a coluna TresGruposKcal

In [332]:
# Regra hierárquica (avaliada na ordem abaixo):
#   1. ProxMeta  : >= 70% da meta em ao menos 50% dos dias internados;
#   2. MetMeta   : não é ProxMeta, mas consumiu >= 50% da meta em ao menos 50%
#                  dos dias (ou seja, a maioria dos dias ficou na faixa de
#                  50% a 69%, ou acima);
#   3. AbaixoMeta: demais pacientes, isto é, consumiu < 50% da meta em mais de
#                  50% dos dias internados.
# A hierarquia garante que cada paciente caia em um único grupo, inclusive
# quando os dias se dividem entre faixas (ex.: 40% dos dias >= 70%,
# 30% entre 50-69% e 30% < 50% -> MetMeta).
condicoes_tres = [
    prop_70_ou_mais >= 0.5,    # ProxMeta
    prop_50_ou_mais >= 0.5,    # MetMeta
]
categorias_tres = ["ProxMeta", "MetMeta"]

df["TresGruposKcal"] = np.select(condicoes_tres, categorias_tres, default="AbaixoMeta")


# Passo 8 - Criar a coluna DoisGruposKcal

In [333]:
#   1. ProxMeta: >= 50% da meta em ao menos 50% dos dias internados;
#   2. MetMeta : demais pacientes (< 50% da meta em mais de 50% dos dias).
df["DoisGruposKcal"] = np.where(prop_50_ou_mais >= 0.5, "ProxMeta", "MetMeta")

# Pacientes sem nenhum dia de consumo registrado (ou sem meta_kcal) ficam sem
# classificação (NaN), em vez de caírem por engano em AbaixoMeta/MetMeta.
sem_dados = dias_internados == 0
df.loc[sem_dados, ["TresGruposKcal", "DoisGruposKcal"]] = np.nan


# Passo 9 - (Opcional) Guardar variáveis auxiliares para conferência

In [334]:
df["dias_internados"] = dias_internados
df["prop_dias_70_ou_mais"] = prop_70_ou_mais.round(3)
df["prop_dias_50_a_69"] = (dias_50_a_69 / dias_internados.replace(0, np.nan)).round(3)
df["prop_dias_abaixo_50"] = (1 - prop_50_ou_mais).round(3)



# Passo 10 - Conferir o resultado

In [335]:
print(df["TresGruposKcal"].value_counts(dropna=False), "\n")
print(df["DoisGruposKcal"].value_counts(dropna=False), "\n")
print(pd.crosstab(df["TresGruposKcal"], df["DoisGruposKcal"]))


TresGruposKcal
AbaixoMeta    27
MetMeta       20
ProxMeta      10
Name: count, dtype: int64 

DoisGruposKcal
ProxMeta    30
MetMeta     27
Name: count, dtype: int64 

DoisGruposKcal  MetMeta  ProxMeta
TresGruposKcal                   
AbaixoMeta           27         0
MetMeta               0        20
ProxMeta              0        10


In [ ]:
# -----------------------------------------------------------------------------
# Passo 11 - (Opcional) Salvar o banco com as novas colunas em excel
# -----------------------------------------------------------------------------
# df.to_excel("data/data_classificado.xlsx", index=False)


# Criar colunas de deltas dos desfechos musculares e faz test-t

1. `AST RF`

2. `AST VL`


In [337]:
# Cria coluna com deltas para desfechos de AST e faz estatistica descritiva agrupando por grupo
df["delta_astrf"] = df["astrf_alta"] - df["astrf_adm"]
df["delta_astvl"] = df["astvl_alta"] - df["astvl_adm"]

In [338]:
import scipy.stats as stats

# Selecionar os valores de delta_astrf de cada grupo
x = df.loc[
    df["DoisGruposKcal"] == "ProxMeta", "delta_astrf"
].dropna()

y = df.loc[
    df["DoisGruposKcal"] == "MetMeta", "delta_astrf"
].dropna()

# Realizar o teste t para amostras independentes
# Welch: não pressupõe igualdade das variâncias entre os grupos
resultado = stats.ttest_ind(x, y, equal_var=False)

# Apresentar estatísticas descritivas e resultado do teste
print("Grupo ProxMeta:")
print(f"N = {len(x)}")
print(f"Média = {x.mean():.2f}")
print(f"Desvio-padrão = {x.std():.2f}")

print("\nGrupo MetMeta:")
print(f"N = {len(y)}")
print(f"Média = {y.mean():.2f}")
print(f"Desvio-padrão = {y.std():.2f}")

print("\nTeste t de Welch:")
print(f"Estatística t = {resultado.statistic:.3f}")
print(f"Valor de p = {resultado.pvalue:.4f}")

Grupo ProxMeta:
N = 30
Média = -0.72
Desvio-padrão = 0.48

Grupo MetMeta:
N = 27
Média = -0.65
Desvio-padrão = 0.45

Teste t de Welch:
Estatística t = -0.611
Valor de p = 0.5435
